In [1]:
import pandas as pd

In [2]:
transitions_df = pd.read_excel("sample_transitions/sample_transitions.xlsx", dtype={'Plant':'str', 'Channel': 'str', 'from_material': 'str', 'to_material': 'str'})


In [3]:
transitions_df.head()

,Plant,Channel,DP_Group_L3,Material_Hierarchy,Material_Desc,from_material,to_material,transition_month,from_share_prev,to_share_next,note
0,3891,10,D05,27AK,ABS ECU,K054359N00,K054345N00,2023-02,1.000000,1.000000,Exact Material_Desc; overlap allowed. Dominant...
1,3891,10,D05,27AK,ABS ECU,K054359N00,K054345N00,2024-03,0.933333,0.723457,Exact Material_Desc; overlap allowed. Dominant...
2,3891,10,D05,27MA,Wheel Speed Sensor,0486000298N00,K144298N49,2020-11,1.000000,1.000000,Exact Material_Desc; overlap allowed. Dominant...
3,3891,10,D05,27YB,Clamping Sleeve,II16774,K115645N00,2020-09,0.750000,1.000000,Exact Material_Desc; overlap allowed. Dominant...
4,3891,10,D11,11HB,Compressor - Electric,K162141B,K162141C,2023-09,0.659794,0.647059,Exact Material_Desc; overlap allowed. Dominant...


In [4]:
transitions_df.columns.tolist()

['Plant',
 'Channel',
 'DP_Group_L3',
 'Material_Hierarchy',
 'Material_Desc',
 'from_material',
 'to_material',
 'transition_month',
 'from_share_prev',
 'to_share_next',
 'note']

In [6]:
# Collect all unique materials (from + to) for each group
group_cols = ['Plant', 'Channel', 'Material_Hierarchy', 'Material_Desc']

# A cleaner approach that avoids the deprecated .append:
transition_materials = (
    pd.concat([transitions_df[group_cols + ['from_material']].rename(columns={'from_material': 'material'}),
               transitions_df[group_cols + ['to_material']].rename(columns={'to_material': 'material'})])
      .groupby(group_cols)['material']
      .apply(lambda x: sorted(x.unique().tolist()))
      .rename('transition_materials_group')
      .reset_index()
)

transitions_df = transitions_df.merge(transition_materials, on=group_cols, how='left')

In [7]:
transitions_df.head()

,Plant,Channel,DP_Group_L3,Material_Hierarchy,Material_Desc,from_material,to_material,transition_month,from_share_prev,to_share_next,note,transition_materials_group
0,3891,10,D05,27AK,ABS ECU,K054359N00,K054345N00,2023-02,1.000000,1.000000,Exact Material_Desc; overlap allowed. Dominant...,"[0486109211N00, 0486109526N49, 0486109532-62, ..."
1,3891,10,D05,27AK,ABS ECU,K054359N00,K054345N00,2024-03,0.933333,0.723457,Exact Material_Desc; overlap allowed. Dominant...,"[0486109211N00, 0486109526N49, 0486109532-62, ..."
2,3891,10,D05,27MA,Wheel Speed Sensor,0486000298N00,K144298N49,2020-11,1.000000,1.000000,Exact Material_Desc; overlap allowed. Dominant...,"[0486000298N00, K144127JAC, K144128N49, K14429..."
3,3891,10,D05,27YB,Clamping Sleeve,II16774,K115645N00,2020-09,0.750000,1.000000,Exact Material_Desc; overlap allowed. Dominant...,"[II16774, K115645N00]"
4,3891,10,D11,11HB,Compressor - Electric,K162141B,K162141C,2023-09,0.659794,0.647059,Exact Material_Desc; overlap allowed. Dominant...,"[K162141B, K162141C]"


In [11]:
transitions_df['transition_materials_group'] = transitions_df['transition_materials_group'].apply(tuple)

transitions_df = transitions_df[['Plant','Channel','Material_Hierarchy', 'Material_Desc','transition_materials_group']].drop_duplicates()

# convert back to lists if needed
transitions_df['transition_materials_group'] = transitions_df['transition_materials_group'].apply(list)

In [15]:
pd.set_option("display.max_colwidth", 2000)
transitions_df.head()

,Plant,Channel,Material_Hierarchy,Material_Desc,transition_materials_group
0,3891,10,27AK,ABS ECU,"[0486109211N00, 0486109526N49, 0486109532-62, 0486109532N49, 0486109532N49JFQD, K054345N00, K054348N00, K054359N00, K054360N00, K054361N00]"
2,3891,10,27MA,Wheel Speed Sensor,"[0486000298N00, K144127JAC, K144128N49, K144298N49, K145033N49]"
3,3891,10,27YB,Clamping Sleeve,"[II16774, K115645N00]"
4,3891,10,11HB,Compressor - Electric,"[K162141B, K162141C]"
5,3891,10,11YF,Unloader Valve Kit,"[K149557K50, K210683]"


Merge with main Dataset

In [20]:
df = pd.read_excel("../data/plant_mat_channel_agg_input.xlsx", dtype={"Plant": 'str', "Material": 'str', "Channel": 'str'})

In [21]:
df.head()

,Plant,Material,Channel,YearMonth,Customer Demand Qty,Customer Demand Value,Working_Days,Material_Desc,Material_Group,Material_Group_Desc,Material_Hierarchy,Material_Hierarchy_Desc,Valuation_Class,Valuation_Class_Desc,Minimum Lot Size,key
0,3891,0204802915,10,2022-01-01,0,0.0,19,Gear shift Unit,PG016,Transmission,16ED,Add-on Gear Actuatio,3100,Trading goods,80,3891_0204802915_10
1,3891,0204802915,10,2022-02-01,0,0.0,16,Gear shift Unit,PG016,Transmission,16ED,Add-on Gear Actuatio,3100,Trading goods,80,3891_0204802915_10
2,3891,0204802915,10,2022-03-01,0,0.0,23,Gear shift Unit,PG016,Transmission,16ED,Add-on Gear Actuatio,3100,Trading goods,80,3891_0204802915_10
3,3891,0204802915,10,2022-04-01,0,0.0,19,Gear shift Unit,PG016,Transmission,16ED,Add-on Gear Actuatio,3100,Trading goods,80,3891_0204802915_10
4,3891,0204802915,10,2022-05-01,0,0.0,19,Gear shift Unit,PG016,Transmission,16ED,Add-on Gear Actuatio,3100,Trading goods,80,3891_0204802915_10


In [22]:
# Merge normally
merged_df = df.merge(
    transitions_df,
    on=['Plant', 'Channel', 'Material_Hierarchy', 'Material_Desc'],
    how='left'
)

# Keep the merge result only where Material is in the transition group
mask = merged_df.apply(
    lambda row: isinstance(row['transition_materials_group'], list)
                and row['Material'] in row['transition_materials_group'],
    axis=1
)

# Null out the transition column where the condition isn't met
merged_df.loc[~mask, 'transition_materials_group'] = None

In [23]:
merged_df.head()

,Plant,Material,Channel,YearMonth,Customer Demand Qty,Customer Demand Value,Working_Days,Material_Desc,Material_Group,Material_Group_Desc,Material_Hierarchy,Material_Hierarchy_Desc,Valuation_Class,Valuation_Class_Desc,Minimum Lot Size,key,transition_materials_group
0,3891,0204802915,10,2022-01-01,0,0.0,19,Gear shift Unit,PG016,Transmission,16ED,Add-on Gear Actuatio,3100,Trading goods,80,3891_0204802915_10,None
1,3891,0204802915,10,2022-02-01,0,0.0,16,Gear shift Unit,PG016,Transmission,16ED,Add-on Gear Actuatio,3100,Trading goods,80,3891_0204802915_10,None
2,3891,0204802915,10,2022-03-01,0,0.0,23,Gear shift Unit,PG016,Transmission,16ED,Add-on Gear Actuatio,3100,Trading goods,80,3891_0204802915_10,None
3,3891,0204802915,10,2022-04-01,0,0.0,19,Gear shift Unit,PG016,Transmission,16ED,Add-on Gear Actuatio,3100,Trading goods,80,3891_0204802915_10,None
4,3891,0204802915,10,2022-05-01,0,0.0,19,Gear shift Unit,PG016,Transmission,16ED,Add-on Gear Actuatio,3100,Trading goods,80,3891_0204802915_10,None


In [24]:
merged_df['transition_materials_group'] = merged_df.apply(
    lambda row: [row['Material']] if row['transition_materials_group'] is None else row['transition_materials_group'],
    axis=1
)

In [30]:
merged_df.groupby(['Plant','Channel'])['Material'].nunique()

Plant  Channel
3891   10          872
       20          136
       30          239
       40          453
3916   10         5688
       20            1
       40            2
Name: Material, dtype: int64

In [32]:
merged_df['transition_materials_group'] = merged_df['transition_materials_group'].apply(tuple)

# now this works directly
merged_df.groupby(['Plant', 'Channel'])['transition_materials_group'].nunique()

# convert back to lists when needed
#merged_df['transition_materials_group'] = merged_df['transition_materials_group'].apply(list)

Plant  Channel
3891   10          782
       20          124
       30          228
       40          446
3916   10         5664
       20            1
       40            2
Name: transition_materials_group, dtype: int64

In [33]:
merged_df.head()

,Plant,Material,Channel,YearMonth,Customer Demand Qty,Customer Demand Value,Working_Days,Material_Desc,Material_Group,Material_Group_Desc,Material_Hierarchy,Material_Hierarchy_Desc,Valuation_Class,Valuation_Class_Desc,Minimum Lot Size,key,transition_materials_group
0,3891,0204802915,10,2022-01-01,0,0.0,19,Gear shift Unit,PG016,Transmission,16ED,Add-on Gear Actuatio,3100,Trading goods,80,3891_0204802915_10,"(0204802915,)"
1,3891,0204802915,10,2022-02-01,0,0.0,16,Gear shift Unit,PG016,Transmission,16ED,Add-on Gear Actuatio,3100,Trading goods,80,3891_0204802915_10,"(0204802915,)"
2,3891,0204802915,10,2022-03-01,0,0.0,23,Gear shift Unit,PG016,Transmission,16ED,Add-on Gear Actuatio,3100,Trading goods,80,3891_0204802915_10,"(0204802915,)"
3,3891,0204802915,10,2022-04-01,0,0.0,19,Gear shift Unit,PG016,Transmission,16ED,Add-on Gear Actuatio,3100,Trading goods,80,3891_0204802915_10,"(0204802915,)"
4,3891,0204802915,10,2022-05-01,0,0.0,19,Gear shift Unit,PG016,Transmission,16ED,Add-on Gear Actuatio,3100,Trading goods,80,3891_0204802915_10,"(0204802915,)"


In [36]:
merged_df = merged_df.to_excel("../data/plant_mat_channel_agg_transitions_input.xlsx", index=False)